In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

### DehazeDDPM 

#### Stage 1: Physical Modelling

They use the FSDGN Model as the backbone of the first stage

In [2]:
class ConvBlock(nn.Module):
    """Standard Convolutional Block with optional Normalization and Activation."""
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=True, 
                 activation='prelu', norm=None):
        super().__init__()
        layers = [nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=bias)]
        
        if norm == 'batch':
            layers.append(nn.BatchNorm2d(out_channels))
        elif norm == 'instance':
            layers.append(nn.InstanceNorm2d(out_channels))

        acts = {
            'relu': nn.ReLU(True),
            'prelu': nn.PReLU(),
            'lrelu': nn.LeakyReLU(0.2, True),
            'tanh': nn.Tanh(),
            'sigmoid': nn.Sigmoid()
        }
        if activation in acts:
            layers.append(acts[activation])
        
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

In [3]:
class DeconvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=4, stride=2, padding=1, bias=True, 
                 activation='prelu', norm=None):
        super().__init__()
        layers = [nn.ConvTranspose2d(in_channels, out_channels, kernel_size, stride, padding, bias=bias)]
        
        if norm == 'batch':
            layers.append(nn.BatchNorm2d(out_channels))
        elif norm == 'instance':
            layers.append(nn.InstanceNorm2d(out_channels))

        acts = {
            'relu': nn.ReLU(True),
            'prelu': nn.PReLU(),
            'lrelu': nn.LeakyReLU(0.2, True),

        }
        if activation in acts:
            layers.append(acts[activation])
            
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


U-Net and Residual Components

In [4]:
class UNetConvBlock(nn.Module):
    def __init__(self, in_chans, out_chans):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_chans, out_chans, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)
        

class UNetUpBlock(nn.Module):
    def __init__(self, in_chans, out_chans, up_mode='upconv'):
        super().__init__()
        if up_mode == 'upconv':
            self.up = nn.ConvTranspose2d(in_chans, out_chans, kernel_size=2, stride=2)
        else:
            self.up = nn.Sequential(
                nn.Upsample(mode='bilinear', scale_factor=2, align_corners=False),
                nn.Conv2d(in_chans, out_chans, kernel_size=1)
            )
        self.conv_block = nn.Sequential(
            nn.Conv2d(out_chans * 2, out_chans, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x, bridge):
        up = self.up(x)
        out = torch.cat([up, bridge], 1)
        return self.conv_block(out)


class SAM(nn.Module):
    """Supervised Attention Module"""
    def __init__(self, n_feat, kernel_size = 1, bias = False):
        super().__init__()
        pad = kernel_size // 2
        self.conv1 = nn.Conv2d(n_feat, n_feat, kernel_size, padding=pad, bias=bias)
        self.conv2 = nn.Conv2d(n_feat, 3, kernel_size, padding=pad, bias=bias)
        self.conv3 = nn.Conv2d(3, n_feat, kernel_size, padding=pad, bias=bias)

    def forward(self, x, x_img):
        x1 = self.conv1(x)
        img = self.conv2(x) + x_img 

        x2 = torch.sigmoid(self.conv3(img))
        return (x1 * x2) + x, img


class ResBlock(nn.Module):
    def __init__(self, channel):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(channel, channel, 3, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(channel, channel, 3, padding=1),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.conv_1x1 = nn.Conv2d(channel, channel, kernel_size=1)

    def forward(self, x):
        return self.layers(x) + self.conv_1x1(x)


class ResBlock_fft_bench(nn.Module):
    def __init__(self, n_feat):
        super().__init__()
        self.main = nn.Conv2d(n_feat, n_feat, kernel_size=3, padding=1) 
        self.mag  = nn.Conv2d(n_feat, n_feat, kernel_size = 1)
        self.pha = nn.Sequential(
            nn.Conv2d(n_feat, n_feat, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(n_feat, n_feat, kernel_size=1),
        )

    def forward(self, x):
        _, _, H, W = x.shape
        fre = torch.fft.rfft2(x, norm = "backward")
        mag, phase = torch.abs(fre), torch.angle(fre)

        mag_out = self.mag(mag)

        # Phase modulation via softmax-weighted attention
        # Apply the attention on the haze correction in the Amplitude sections
        mag_res = mag_out - mag
        weight = F.softmax(F.adaptive_avg_pool2d(mag_res, (1, 1)), dim=1)
        # I know which frequency channels needed the most amplitude correction (from weight). 
        # Phase errors are probably concentrated in those same channels. 
        # So scale the phase by that attention before trying to correct it 
        # — this tells the network where to focus.
        phase_out = self.pha(phase * weight) + phase

        # After finding the correct phase and amplitude, we can recover back via 
        # inverse fourier transform (IVF)
        real = mag_out * torch.cos(phase_out)
        imag = mag_out * torch.sin(phase_out)
        fre_out = torch.complex(real, imag)
        y = torch.fft.irfft2(fre_out, s=(H, W), norm='backward')

        return self.main(x) + y

#### Encoder and Decoder in the FSDGN model (originally from the MSBDN model)

The core reason for using these specialized blocks is information preservation through multi-scale refinement.

1. **Why we need MDCBlocks**

Ordinary downsampling is a "lossy" process. When you reduce an image's size, you permanently discard high-frequency details (textures, sharp edges, and fine colors). In a standard U-Net, we try to recover this with skip connections, but that is often not enough for "ill-posed" problems where the input is severely degraded.


The MDCBlock (Multi-scale Dual-branch Fusion) solves this by:

- **Sequential Back-Projection (Iter1)**: It doesn't just downsample; it projects the feature down, then immediately tries to project it back up to see what was lost. This "residual" (the difference between what it expected and what it got) is used to correct the feature map, ensuring that even after downsampling, the "essence" of the high-resolution detail is preserved.


- **Iterative Refinement (Iter2)**: It uses multiple iterations to "memorize" and align distributions. Instead of a one-step mapping, it treats the scale change as a feedback loop, which is much more robust for complex non-homogeneous distributions.


2. **What happens if you use ordinary downsampling?**

If you replace these blocks with ordinary downsampling/upsampling layers:


- **Massive Information Loss**: In dense-haze scenarios, very little original information remains. Ordinary downsampling further reduces this limited signal-to-noise ratio, making it nearly impossible for the decoder to reconstruct the scene accurately.


- **Blurry Reconstructions**: Ordinary layers tend to produce "average" results that lack sharp high-frequency details. This leads to blurry, unrealistic images because the network isn't "punished" or "corrected" for what it loses during the downsampling process.

- **Poor Perceptual Quality**: You might get okay "Distortion" scores (like PSNR), but the "Perceptual" quality (like FID or LPIPS) will drop significantly because the fine textures that make an image look "real" were discarded early in the encoder.

- **Distribution Deviation**: In tasks like diffusion-based dehazing, the model needs the input distribution to be as close to the "clear" image distribution as possible. Ordinary downsampling causes the distribution to deviate further, making it harder for the model to "memorize" the clear data distribution.

*Note*: 
You can specifically isolate the "intelligence" of your blocks by comparing them against standard baselines:


- Ablation Study (Standard vs. MDC): Replace your MDCBlocks with ordinary downsampling/upsampling (like nn.MaxPool2d or standard nn.Conv2d with stride 2) and measure the drop in LPIPS.


- Scale Alignment Test: Test the effectiveness of your iter2 mode (interpolation-based feedback) by checking for artifacts in regions where the non-homogeneous haze is densest. The MDCBlock is designed to "memorize" distribution better in these hard regions.


- Transmission Map Guidance: In your Encoder, ensure the ft_high_list (skip connections) effectively guides the low-res input. You can visualize the intermediate trmap (transmission map) to see if it correctly identifies high-density haze regions in the NH-HAZE samples.

In [22]:
class Decoder_MDCBlock(torch.nn.Module):
    """
    Refined Multi-scale Fusion for the Decoder path.
    Fuses High-resolution features with a list of Low-resolution feature maps.
    """
    def __init__(self, num_filter, num_ft, num, kernel_size=4, stride=2, padding=1, 
                 bias=True, activation='prelu', norm=None, mode='iter2'):
        super(Decoder_MDCBlock, self).__init__()
        self.mode = mode
        self.num_ft = num_ft - 1
        self.down_convs = nn.ModuleList()
        self.up_convs = nn.ModuleList()

        in_ch = num_filter
        for i in range(self.num_ft):
            out_ch = in_ch + 2 ** (num + i)
            
            self.down_convs.append(
                ConvBlock(in_ch, out_ch, kernel_size, stride, padding, bias, activation)
            )
            self.up_convs.append(
                DeconvBlock(out_ch, in_ch, kernel_size, stride, padding, bias, activation)
            )
            in_ch = out_ch

    def _forward_iter1(self, ft_h, ft_l_list):
        """
        Sequential back-projection logic.
        Matches the logic used in DBPN (Deep Back-Projection Networks).
        """
        history = []
        # Donwward pass: Track the state of the high-res feature at different scale
        for i in range(len(ft_l_list)):
            history.append(ft_h)
            idx = max(0, self.num_ft - len(ft_l_list) + i)
            ft_h = self.down_convs[idx](ft_h)

        # Upward fusion pass: Calculate residual error between scales
        fusion = ft_h
        for i in range(len(ft_l_list)):
            residual = fusion - ft_l_list[i]
            idx = max(0, self.num_ft - i - 1)
            # Apply up-conv to residual and add back the historical high-res feature
            fusion = self.up_convs[idx](residual) + history[len(ft_l_list) - i - 1]

        return fusion


    def _forward_iter2(self, ft_h, ft_l_list):
        """Interpolation-based feedback mode."""
        fusion = ft_h

        for i in range(len(ft_l_list)):
            temp = fusion 

            # 1. Project Down: Progressively reduce resolution
            # We only project down as many times as there are levels in the list
            num_steps = self.num_ft - i
            for j in range(num_steps):
                temp = self.down_convs[j](temp)

            # 2. Align spatial size and compute error
            target_shape = ft_l_list[i].shape[-2:]
            temp = F.interpolate(temp, size = target_shape, mode = 'bilinear', align_corners=False)
            error = temp - ft_l_list[i]

            # 3. Project error backup to high-resolution
            for j in range(num_steps):
                # Reverse idx for up-sampling
                up_idx = num_steps - j - 1
                error = self.up_convs[up_idx](error)
            
            # 4. Update the high-resolution fusion feature
            error_shape = error.shape[-2:]
            fusion_resized = F.interpolate(fusion, size = error_shape, mode='bilinear', align_corners=False)
            fusion = fusion_resized + error
            
        return fusion        
    
 
    def forward(self, ft_high, ft_low_list):
        """
        ft_high: High-resolution input feature map
        ft_low_list: List of low-resolution skip-connection features
        """
        # return ft_fusion
        if self.mode in ['iter1', 'conv']:
            return self._forward_iter1(ft_high, ft_low_list)
        elif self.mode in ['iter2']:
            return self._forward_iter2(ft_high, ft_low_list)

        else:
            raise ValueError(f"Mode '{self.mode}' is not supported. Use 'iter1' or 'iter2'.")
  

In [29]:
class Encoder_MDCBlock(torch.nn.Module):
    """
    Refined Multi-scale Fusion for the Encoder path.
    Fuses Low-resolution features with a list of High-resolution feature maps.
    """
    def __init__(self, num_filter, num_ft, kernel_size=4, stride=2, padding=1, 
                 bias=True, activation='prelu', norm=None, mode='iter2'):
        super(Encoder_MDCBlock, self).__init__()
        self.mode = mode
        self.num_ft = num_ft - 1
        self.up_convs = nn.ModuleList()
        self.down_convs = nn.ModuleList()
        
        in_ch = num_filter 
        for i in range(self.num_ft):
            # Channels decrease as resolution increases in the Encoder
            out_ch = in_ch - 2 ** (num_ft - i)
            
            self.up_convs.append(
                DeconvBlock(in_ch, out_ch, kernel_size, stride, padding, bias, activation)
            )
            self.down_convs.append(
                ConvBlock(out_ch, in_ch, kernel_size, stride, padding, bias, activation)
            )
            in_ch = out_ch

    def _forward_iter1(self, ft_l, ft_h_list):
        """
        Sequential back-projection logic for Encoder.
        """
        history = []
        n = len(ft_h_list)
        
        # Upward pass: Moving from Low-res input to High-res scales
        for i in range(n):
            history.append(ft_l)
            idx = max(0, self.num_ft - n + i)
            ft_l = self.up_convs[idx](ft_l)

        # Downward fusion pass: Calculate residual error in high-res space
        fusion = ft_l
        for i in range(n):
            residual = fusion - ft_h_list[i]
            idx = max(0, self.num_ft - i - 1)
            # Apply down-conv to residual and add back the historical low-res state
            fusion = self.down_convs[idx](residual) + history[n - i - 1]

        return fusion

    def _forward_iter2(self, ft_l, ft_h_list):
        """Interpolation-based feedback mode for Encoder."""
        fusion = ft_l
        n = len(ft_h_list)

        for i in range(n):
            temp = fusion 

            # 1. Project Up: Increase resolution to reach target skip-connection scale
            num_steps = self.num_ft - i
            for j in range(num_steps):
                temp = self.up_convs[j](temp)
            # 2. Align spatial size and compute error
            target_shape = ft_h_list[i].shape[-2:]
            if temp.shape[-2:] != target_shape:
                temp = F.interpolate(temp, size=target_shape, mode='bilinear', align_corners=False)
            error = temp - ft_h_list[i]

            # 3. Project error back down to low-resolution
            for j in range(num_steps):
                # Reverse idx for down-sampling
                down_idx = num_steps - j - 1
                error = self.down_convs[down_idx](error)
            
            # 4. Update the low-resolution fusion feature
            if fusion.shape[-2:] != error.shape[-2:]:
                fusion = F.interpolate(fusion, size=error.shape[-2:], mode='bilinear', align_corners=False)
            
            fusion = fusion + error
            
        return fusion 
    
    def forward(self, ft_low, ft_high_list):
        """
        ft_low: Low-resolution input feature map
        ft_high_list: List of high-resolution skip-connection features
        """
        if self.mode in ['iter1', 'conv']:
            return self._forward_iter1(ft_low, ft_high_list)
        elif self.mode == 'iter2':
            return self._forward_iter2(ft_low, ft_high_list)
        else:
            raise ValueError(f"Mode '{self.mode}' is not supported. Use 'iter1' or 'iter2'.")

## FFT - branch

In [30]:
class ResBlock_fft_bench(nn.Module):
    def __init__(self, n_feat):
        super().__init__()
        self.main = nn.Conv2d(n_feat, n_feat, kernel_size=3, padding=1) 
        self.mag  = nn.Conv2d(n_feat, n_feat, kernel_size = 1)
        self.pha = nn.Sequential(
            nn.Conv2d(n_feat, n_feat, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(n_feat, n_feat, kernel_size=1),
        )

    def forward(self, x):
        _, _, H, W = x.shape
        fre = torch.fft.rfft2(x, norm = "backward")
        mag, phase = torch.abs(fre), torch.angle(fre)

        mag_out = self.mag(mag)

        # Phase modulation via softmax-weighted attention
        # Apply the attention on the haze correction in the Amplitude sections
        mag_res = mag_out - mag
        weight = F.softmax(F.adaptive_avg_pool2d(mag_res, (1, 1)), dim=1)
        # I know which frequency channels needed the most amplitude correction (from weight). 
        # Phase errors are probably concentrated in those same channels. 
        # So scale the phase by that attention before trying to correct it 
        # — this tells the network where to focus.
        phase_out = self.pha(phase * weight) + phase

        # After finding the correct phase and amplitude, we can recover back via 
        # inverse fourier transform (IVF)
        real = mag_out * torch.cos(phase_out)
        imag = mag_out * torch.sin(phase_out)
        fre_out = torch.complex(real, imag)
        y = torch.fft.irfft2(fre_out, s=(H, W), norm='backward')

        return self.main(x) + y


## Subnetwork for predicting the physical properties

In this case it would be the Global Atmospheric Light

In [31]:
class BlockUNet1(nn.Module):
    def __init__(self, in_channels, out_channels, upsample=False, relu=False, drop=False, bn=True):
        super(BlockUNet1, self).__init__()

        self.conv = nn.Conv2d(in_channels, out_channels, 4, 2, 1, bias=False)
        self.deconv = nn.ConvTranspose2d(in_channels, out_channels, 4, 2, 1, bias=False)

        self.dropout = nn.Dropout2d(0.5)
        self.batch = nn.InstanceNorm2d(out_channels)

        self.upsample = upsample
        self.relu = relu
        self.drop = drop
        self.bn = bn

    def forward(self, x):
        if self.relu == True:
            y = F.relu(x)
        elif self.relu == False:
            y = F.leaky_relu(x, 0.2)
        if self.upsample == True:
            y = self.deconv(y)
            if self.bn == True:
                y = self.batch(y)
            if self.drop == True:
                y = self.dropout(y)

        elif self.upsample == False:
            y = self.conv(y)
            if self.bn == True:
                if y.shape[2] == 1:
                    y = y
                else:
                    y = self.batch(y)
            if self.drop == True:
                y = self.dropout(y)

        return y

In [32]:
class G2(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(G2, self).__init__()

        self.conv = nn.Conv2d(in_channels, 8, 4, 2, 1, bias=False)
        self.layer1 = BlockUNet1(8, 16)
        self.layer2 = BlockUNet1(16, 32)

        self.dlayer2 = BlockUNet1(32, 16, upsample = True, relu = True, drop = True, bn = False)
        self.dlayer1 = BlockUNet1(32, 8, upsample = True, relu =  True)
        self.relu = nn.ReLU()
        self.dconv = nn.ConvTranspose2d(16, out_channels, 4, 2, 1, bias=False)
        self.lrelu = nn.LeakyReLU(0.2)

    def forward(self, x):
        y1 = self.conv(x)
        y2 = self.layer1(y1)
        y3 = self.layer2(y2)

        dy3 = self.dlayer2(y3)
        concat2 = torch.cat([dy3, y2], 1)
        dy2 = self.dlayer1(concat2)
        concat1 = torch.cat([dy2, y1], 1)
        out = self.relu(concat1)
        out = self.dconv(out)
        out = self.lrelu(out)

        return F.avg_pool2d(out, (out.shape[2], out.shape[3]))


### FSDGN - Model

In [33]:
### MDC Blocks

class FSDGN(nn.Module):
    def __init__(self, num_in_ch=3, base_channel=16, up_mode='upconv', bias=False):
        super().__init__()
        
        # Channels: [16, 20, 28, 44, 76]
        chs = [base_channel, 20, 28, 44, 76]
        # Channels sequence for Encoder -> Bottleneck -> Decoder
        # E.g [16, 20, 28, 44, 76, 44, 28, 20, 16] 
        chs_enc_dec = chs + chs[-2::-1]

        # ------------------- STAGE 1 (Frequency / Global Branch) -------------------
        self.enc_convs = nn.ModuleList([nn.Conv2d(num_in_ch, chs[0], 3, 1, 1)])
        self.enc_convs.extend([UNetConvBlock(chs[i], chs[i + 1]) for i in range(4)])
        self.dec_convs = nn.ModuleList([UNetUpBlock(chs[i + 1], chs[i], up_mode) for i in range(4)])
        
        # Frequency and Residual Blocks
        self.res_blocks = nn.ModuleList([ResBlock(c) for c in chs_enc_dec])
        self.fft_blocks = nn.ModuleList([ResBlock_fft_bench(c) for c in chs_enc_dec])
        
        # Fusion Blocks (
        self.enc_fusions = nn.ModuleList([
            Encoder_MDCBlock(chs[1], 2), Encoder_MDCBlock(chs[2], 3),
            Encoder_MDCBlock(chs[3], 4), Encoder_MDCBlock(chs[4], 5)
        ])
        self.dec_fusions = nn.ModuleList([
            Decoder_MDCBlock(chs[3], 2, 5), Decoder_MDCBlock(chs[2], 3, 4),
            Decoder_MDCBlock(chs[1], 4, 3), Decoder_MDCBlock(chs[0], 5, 2)
        ])

        # ------------------- STAGE 2 (Spatial / Local Branch) -------------------
        self.enc_convs2 = nn.ModuleList([nn.Conv2d(num_in_ch, chs[0], 3, 1, 1)])
        self.enc_convs2.extend([UNetConvBlock(chs[i], chs[i+1]) for i in range(4)])
        self.dec_convs2 = nn.ModuleList([UNetUpBlock(chs[i+1], chs[i], up_mode) for i in range(4)])
        self.res_blocks2 = nn.ModuleList([ResBlock(c) for c in chs_enc_dec])
        
        self.enc_fusions2 = nn.ModuleList([
            Encoder_MDCBlock(chs[1], 2), Encoder_MDCBlock(chs[2], 3),
            Encoder_MDCBlock(chs[3], 4), Encoder_MDCBlock(chs[4], 5)
        ])
        self.dec_fusions2 = nn.ModuleList([
            Decoder_MDCBlock(chs[3], 2, 5), Decoder_MDCBlock(chs[2], 3, 4),
            Decoder_MDCBlock(chs[1], 4, 3), Decoder_MDCBlock(chs[0], 5, 2)
        ])

        # Cross-Stage Feature Fusion (CSFF)
        self.csff_enc = nn.ModuleList([nn.Conv2d(c, c, 1, bias=bias) for c in chs[:-1]])
        self.csff_dec = nn.ModuleList([nn.Conv2d(c, c, 1, bias=bias) for c in chs[:-1]])

        self.sam = SAM(chs[0], kernel_size=1)
        self.concat = nn.Conv2d(chs[0] * 2, chs[0], 3, padding=1)

        # ------------------- PHYSICS COMPONENTS (ASM) -------------------
        # J-Net: Last layer for clean image
        self.last = nn.Conv2d(chs[0], num_in_ch, kernel_size=1)

        # # A-Net: Global Atmospheric Light
        self.ANet = G2(3, 3)

        # T-Net: Transmission Map Estimation (derived from Stage 2 features)
        self.conv_T_1 = nn.Conv2d(base_channel, base_channel, 3, 1, 1, bias=False)
        self.conv_T_2 = nn.Conv2d(base_channel, 1, 3, 1, 1, bias=False)

    def forward(self, x):
        identity = x
        
        # ================= STAGE 1: Frequency Guided Branch =================
        enc_feats = []
        out = self.enc_convs[0](x)
        out = self.fft_blocks[0](self.res_blocks[0](out))
        enc_feats.append(out)

        # Stage 1 Encoder 
        for i in range(4):
            out = self.enc_convs[i + 1](out)
            out = self.enc_fusions[i](out, enc_feats)
            out = self.fft_blocks[i + 1](self.res_blocks[i + 1](out))
            enc_feats.append(out)

        # Stage 1 Decoder
        dec_feats = [enc_feats[-1]]
        curr_out = enc_feats[-1]
        for i in range(4):
            idx = 3 - i # 3, 2, 1, 0 (Reversing back up the U-Net)
            # print("Curr_out: ", curr_out.shape)
            # print(f"Enc_feats at {idx}: {enc_feats[idx].shape}")
            
            curr_out = self.dec_convs[idx](curr_out, enc_feats[idx])
            
            # 5+i maps to [44, 28, 20, 16] in the chs_enc_dec list
            curr_out = self.res_blocks[5 + i](curr_out)
            curr_out = self.fft_blocks[5 + i](curr_out)
            curr_out = self.dec_fusions[i](curr_out, dec_feats)
            
            dec_feats.append(curr_out)

        sam_feats, stage1_img = self.sam(dec_feats[-1], identity)

        # ================= STAGE 2: Spatial Guided Branch =================
        enc_feats2 = []
        
        out_2 = self.enc_convs2[0](identity)
        y_concat = self.concat(torch.cat([out_2, sam_feats], dim = 1))
        
        # CSFF from Stage 1: dec_feats[-1] is the last output of Stage 1 decoder
        y = self.res_blocks2[0](y_concat)
        y = y + self.csff_enc[0](enc_feats[0]) + self.csff_dec[0](dec_feats[-1])
        enc_feats2.append(y)

        # --- Encoder Stage 2 ---
        for i in range(4):
            y = self.enc_convs2[i + 1](y)
            y = self.enc_fusions2[i](y, enc_feats2)
            y = self.res_blocks2[i + 1](y)
            if i < 3: # Apply CSFF skip connections
                y = y + self.csff_enc[i + 1](enc_feats[i + 1]) + \
                        self.csff_dec[i + 1](dec_feats[-(i + 2)])
            enc_feats2.append(y)

        # Decoder Stage 2
        dec_feats2 = [enc_feats2[-1]]
        curr_y = enc_feats2[-1]
        for i in range(4):
            idx = 3 - i
            curr_y = self.dec_convs2[idx](curr_y, enc_feats2[idx])
            curr_y = self.dec_fusions2[i](self.res_blocks2[5 + i](curr_y), dec_feats2)
            dec_feats2.append(curr_y)

        # ================= PHYSICS MODELING (ASM) =================
        # 1. Prediction of Clean Image J
        out_J = torch.clamp(self.last(dec_feats2[-1]), 0, 1)
        
        # 2. Prediction of Transmission Map T
        out_T = self.conv_T_1(dec_feats2[-1])
        out_T = torch.sigmoid(self.conv_T_2(out_T)) # Using Sigmoid for [0,1] range
        
        # 3. Prediction of Atmospheric Light A
        out_A = torch.sigmoid(self.ANet(identity)) # Fixed xcopy -> identity
        
        # 4. Reconstruction of Hazy Image I: I = J*T + A(1-T)
        out_I = out_T * out_J + (1 - out_T) * out_A
        
        # Intermediate outputs for supervision
        stage1_img = torch.clamp(stage1_img, 0, 1)

        return out_J, stage1_img, out_T, out_A, out_I

In [34]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_1 = FSDGN().to(device)
input = torch.randn(1, 3, 256, 256).to(device)
out_J, _, out_T, out_A, out_I = model_1(input)

print(f"Shape of output J is: {out_J.shape}")

Shape of output J is: torch.Size([1, 3, 256, 256])


#### Stage 2: Diffusion Process


UNet Section for the Dehazing Function

In [3]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from inspect import isfunction

Helper Functions

In [4]:
def exists(x):
    return x is not None

def default(val, d):
    if exists(val):
        return val
    return d() if isfunction(d) else d

class Swish(nn.Module):
    """Swish activation function, commonly used in diffusion models."""
    def forward(self, x):
        return x * torch.sigmoid(x)

Model

Time Embedding

In [5]:
class TimeEmbedding(nn.Module):
    """
    Encodes the diffusion timestep `t` into a high-dimensional continuous vector.
    This allows the network to know which noise level it is currently trying to predict.
    """
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        inv_freq = torch.exp(
            torch.arange(0, dim, 2, dtype=torch.float32) * (-math.log(10000) / dim)
        )
        self.register_buffer("inv_freq", inv_freq)

    def forward(self, input):
        shape = input.shape
        sinusoid_in = torch.ger(input.view(-1).float(), self.inv_freq)
        pos_emb = torch.cat([sinusoid_in.sin(), sinusoid_in.cos()], dim=-1)
        return pos_emb.view(*shape, self.dim)

Spatial Transformation

In [6]:
class Upsample(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="nearest")
        self.conv = nn.Conv2d(dim, dim, 3, padding=1)

    def forward(self, x):
        return self.conv(self.up(x))

class Downsample(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.conv = nn.Conv2d(dim, dim, 3, 2, 1) # Stride 2 for downsampling

    def forward(self, x):
        return self.conv(x)

Core Building Blocks

In [7]:
class Block(nn.Module):
    """Standard Convolutional Block with GroupNorm and Swish."""
    def __init__(self, dim, dim_out, groups=32, dropout=0):
        super().__init__()
        self.block = nn.Sequential(
            nn.GroupNorm(groups, dim),
            Swish(),
            nn.Dropout(dropout) if dropout != 0 else nn.Identity(),
            nn.Conv2d(dim, dim_out, 3, padding=1)
        )
    def forward(self, x):
        return self.block(x)

class ResnetBlock(nn.Module):
    """Residual block that injects the Time Embedding into the spatial features."""
    def __init__(self, dim, dim_out, time_emb_dim=None, dropout=0, norm_groups=32):
        super().__init__()
        # Time embedding projection layer
        self.mlp = nn.Sequential(
            Swish(),
            nn.Linear(time_emb_dim, dim_out)
        ) if exists(time_emb_dim) else None

        self.block1 = Block(dim, dim_out, groups=norm_groups)
        self.block2 = Block(dim_out, dim_out, groups=norm_groups, dropout=dropout)
        
        # Skip connection alignment
        self.res_conv = nn.Conv2d(dim, dim_out, 1) if dim != dim_out else nn.Identity()

    def forward(self, x, time_emb):
        h = self.block1(x)
        # Inject time embedding by adding it to the feature map
        if exists(self.mlp):
            h += self.mlp(time_emb)[:, :, None, None]
            
        h = self.block2(h)
        return h + self.res_conv(x)

In [8]:
class SelfAttention(nn.Module):
    """Multi-head Self-Attention to capture global dependencies in hazy images."""
    def __init__(self, in_channel, n_head=1, norm_groups=32):
        super().__init__()
        self.n_head = n_head
        self.norm = nn.GroupNorm(norm_groups, in_channel)
        self.qkv = nn.Conv2d(in_channel, in_channel * 3, 1, bias=False)
        self.out = nn.Conv2d(in_channel, in_channel, 1)

    def forward(self, input):
        batch, channel, height, width = input.shape
        n_head = self.n_head
        head_dim = channel // n_head

        norm = self.norm(input)
        qkv = self.qkv(norm).view(batch, n_head, head_dim * 3, height, width)
        query, key, value = qkv.chunk(3, dim=2) 

        # Calculate attention scores
        attn = torch.einsum("bnchw, bncyx -> bnhwyx", query, key).contiguous() / math.sqrt(channel)
        attn = attn.view(batch, n_head, height, width, -1)
        attn = torch.softmax(attn, -1)
        attn = attn.view(batch, n_head, height, width, height, width)

        # Apply attention to values
        out = torch.einsum("bnhwyx, bncyx -> bnchw", attn, value).contiguous()
        out = self.out(out.view(batch, channel, height, width))

        return out + input

In [9]:
class ResnetBlocWithAttn(nn.Module):
    """Combines ResNet block and optional Self-Attention."""
    def __init__(self, dim, dim_out, *, time_emb_dim=None, norm_groups=32, dropout=0, with_attn=False):
        super().__init__()
        self.with_attn = with_attn
        self.res_block = ResnetBlock(dim, dim_out, time_emb_dim, dropout=dropout, norm_groups=norm_groups)
        if with_attn:
            self.attn = SelfAttention(dim_out, norm_groups=norm_groups)

    def forward(self, x, time_emb):
        x = self.res_block(x, time_emb)
        if self.with_attn:
            x = self.attn(x)
        return x

Main Architecture with UNet

In [13]:
# ==========================================
# 5. MAIN U-NET ARCHITECTURE
# ==========================================
class UNet(nn.Module):
    def __init__(
        self,
        in_channel=6,         # 3 (noisy) + 3 (J)  = 7 for DehazeDDPM
        out_channel=3,        # Predicts the 3-channel added noise
        inner_channel=32,
        norm_groups=32,
        channel_mults=(1, 2, 4, 8, 8),
        attn_res=(8,),     # Resolutions to apply self-attention
        res_blocks=3,
        dropout=0,
        with_time_emb=True,
        image_size=128
    ):
        super().__init__()

        # --- Time Embedding MLP ---
        if with_time_emb:
            time_dim = inner_channel
            self.time_mlp = nn.Sequential(
                TimeEmbedding(inner_channel),
                nn.Linear(inner_channel, inner_channel * 4),
                Swish(),
                nn.Linear(inner_channel * 4, inner_channel)
            )
        else:
            time_dim = None
            self.time_mlp = None

        num_mults = len(channel_mults)
        pre_channel = inner_channel
        feat_channels = [pre_channel]
        now_res = image_size
        
        # --- Encoder (Downsampling) ---
        downs = [nn.Conv2d(in_channel, inner_channel, kernel_size=3, padding=1)]
        for ind in range(num_mults):
            is_last = (ind == num_mults - 1)
            use_attn = (now_res in attn_res)
            channel_mult = inner_channel * channel_mults[ind]
            
            for _ in range(0, res_blocks):
                downs.append(ResnetBlocWithAttn(
                    pre_channel, channel_mult, time_emb_dim=time_dim, 
                    norm_groups=norm_groups, dropout=dropout, with_attn=use_attn
                ))
                feat_channels.append(channel_mult)
                pre_channel = channel_mult
                
            if not is_last:
                downs.append(Downsample(pre_channel))
                feat_channels.append(pre_channel)
                now_res = now_res // 2
        self.downs = nn.ModuleList(downs)

        # --- Bottleneck (Middle) ---
        self.mid = nn.ModuleList([
            ResnetBlocWithAttn(pre_channel, pre_channel, time_emb_dim=time_dim, 
                               norm_groups=norm_groups, dropout=dropout, with_attn=True),
            ResnetBlocWithAttn(pre_channel, pre_channel, time_emb_dim=time_dim, 
                               norm_groups=norm_groups, dropout=dropout, with_attn=False)
        ])

        # --- Decoder (Upsampling) ---
        ups = []
        for ind in reversed(range(num_mults)):
            is_last = (ind < 1)
            use_attn = (now_res in attn_res)
            channel_mult = inner_channel * channel_mults[ind]
            
            for _ in range(0, res_blocks + 1):
                # +feat_channels.pop() handles the Skip Connection dimension
                ups.append(ResnetBlocWithAttn(
                    pre_channel + feat_channels.pop(), channel_mult, time_emb_dim=time_dim, 
                    dropout=dropout, norm_groups=norm_groups, with_attn=use_attn
                ))
                pre_channel = channel_mult
                
            if not is_last:
                ups.append(Upsample(pre_channel))
                now_res = now_res * 2

        self.ups = nn.ModuleList(ups)
        self.final_conv = Block(pre_channel, default(out_channel, in_channel), groups=norm_groups)

    def forward(self, x, time):
        # 1. Generate time embedding
        t = self.time_mlp(time) if exists(self.time_mlp) else None

        # 2. Pass through Encoder, saving skip connections
        feats = []
        for layer in self.downs:
            if isinstance(layer, ResnetBlocWithAttn):
                x = layer(x, t)
            else:
                x = layer(x)
            feats.append(x)

        # 3. Pass through Bottleneck
        for layer in self.mid:
            if isinstance(layer, ResnetBlocWithAttn):
                x = layer(x, t)
            else:
                x = layer(x)

        # 4. Pass through Decoder, concatenating skip connections
        for layer in self.ups:
            if isinstance(layer, ResnetBlocWithAttn):
                # Torch.cat applies the U-Net skip connection
                x = layer(torch.cat((x, feats.pop()), dim=1), t)
            else:
                x = layer(x)

        # 5. Output predicted noise
        return self.final_conv(x)

In [14]:
dehazeddpm = UNet()

print(dehazeddpm)

UNet(
  (time_mlp): Sequential(
    (0): TimeEmbedding()
    (1): Linear(in_features=32, out_features=128, bias=True)
    (2): Swish()
    (3): Linear(in_features=128, out_features=32, bias=True)
  )
  (downs): ModuleList(
    (0): Conv2d(6, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1-3): 3 x ResnetBlocWithAttn(
      (res_block): ResnetBlock(
        (mlp): Sequential(
          (0): Swish()
          (1): Linear(in_features=32, out_features=32, bias=True)
        )
        (block1): Block(
          (block): Sequential(
            (0): GroupNorm(32, 32, eps=1e-05, affine=True)
            (1): Swish()
            (2): Identity()
            (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          )
        )
        (block2): Block(
          (block): Sequential(
            (0): GroupNorm(32, 32, eps=1e-05, affine=True)
            (1): Swish()
            (2): Identity()
            (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), pa

Diffusion Step of the Model

In [15]:
import math
import torch
from torch import nn
import torch.nn.functional as F
import numpy as np
from tqdm.notebook import tqdm
from functools import partial

In [19]:
def _warmup_beta(linear_start, linear_end, n_timestep, warmup_frac):
    """
    Creates a beta schedule that linearly increases for a warmup period, 
    then remains constant at the 'linear_end' value.
    """
    betas = linear_end * np.ones(n_timestep, dtype=np.float64)
    warmup_time = int(n_timestep * warmup_frac)
    betas[:warmup_time] = np.linspace(
        linear_start, linear_end, warmup_time, dtype=np.float64)
    return betas

def make_beta_schedule(schedule, n_timestep, linear_start=1e-4, linear_end=2e-2, cosine_s=8e-3):
    """
    Generates a variance (beta) schedule for the diffusion process.
    
    Args:
        schedule (str): The type of schedule ('linear', 'cosine', 'quad', etc.)
        n_timestep (int): Total number of diffusion steps (T).
        linear_start (float): Starting variance at t=0.
        linear_end (float): Ending variance at t=T (used by linear, quad, warmup).
        cosine_s (float): Offset for the cosine schedule to prevent singularities.
        
    Returns:
        betas (np.ndarray or torch.Tensor): 1D array of beta values.
    """
    if schedule == 'quad':
        # Quadratic schedule: Starts slow, accelerates towards the end
        betas = np.linspace(linear_start ** 0.5, linear_end ** 0.5,
                            n_timestep, dtype=np.float64) ** 2
        
    elif schedule == 'linear':
        # Standard linear schedule (used in original DDPM)
        betas = np.linspace(linear_start, linear_end,
                            n_timestep, dtype=np.float64)
        
    elif schedule == 'warmup10':
        # Linear increase for the first 10% of steps, then constant
        betas = _warmup_beta(linear_start, linear_end,
                             n_timestep, 0.1)
        
    elif schedule == 'warmup50':
        # Linear increase for the first 50% of steps, then constant
        betas = _warmup_beta(linear_start, linear_end,
                             n_timestep, 0.5)
        
    elif schedule == 'const':
        # Constant noise addition at every step
        betas = linear_end * np.ones(n_timestep, dtype=np.float64)
        
    elif schedule == 'jsd':  # 1/T, 1/(T-1), 1/(T-2), ..., 1
        # Jensen-Shannon Divergence schedule: 1/T, 1/(T-1), 1/(T-2), ..., 1
        betas = 1. / np.linspace(n_timestep,
                                 1, n_timestep, dtype=np.float64)
        
    elif schedule == "cosine":
        # Cosine schedule (Nichol and Dhariwal, 2021)
        # Prevents destroying information too quickly in the forward process
        timesteps = (
            torch.arange(n_timestep + 1, dtype=torch.float64) /
            n_timestep + cosine_s
        )
        alphas = timesteps / (1 + cosine_s) * math.pi / 2
        alphas = torch.cos(alphas).pow(2)
        alphas = alphas / alphas[0]
        betas = 1 - alphas[1:] / alphas[:-1]
        betas = betas.clamp(max=0.999) # Clamp to prevent numerical instability
    else:
        raise NotImplementedError(schedule)
    return betas


In [17]:
def exists(x):
    return x is not None


def default(val, d):
    if exists(val):
        return val
    return d() if isfunction(d) else d


In [ ]:
# --- Gaussian Diffusion Class ---
class GaussianDiffusion(nn.Module):
    def __init__(
        self,
        denoise_fn, 
        image_size, 
        channels = 3, 
        loss_type = 'l1',
        conditional = True, 
        schedule_opt = None, 
        freq_weight = 0.01     # Weight for the Frequency Prior Optimization
    ):
        super().__init__()
        self.channels = channels
        self.image_size = image_size
        self.denoise_fn = denoise_fn
        self.loss_type = loss_type
        self.conditional = conditional
        self.freq_weight = freq_weight
        
        if schedule_opt is not None:
            pass

        
    def set_loss(self, device):
        if self.loss_type == 'l1':
            self.loss_func = nn.L1Loss(reduction='mean').to(device)
        elif self.loss_type == 'l2':
            self.loss_func = nn.MSELoss(reduction='mean').to(device)
        else:
            raise NotImplementedError()


    def set_new_noise_scheduler(self, schedule_opt, device):
        to_torch = partial(torch.tensor, dtype=torch.float32, device=device)

        betas = make_beta_schedule(
            schedule=schedule_opt['schedule'],
            n_timestep=schedule_opt['n_timestep'],
            linear_start=schedule_opt['linear_start'],
            linear_end=schedule_opt['linear_end'])

        betas = betas.detach().cpu().numpy() if isinstance(betas, torch.Tensor) else betas 
        alphas = 1. - betas 
        alphas_cumprod = np.cumprod(alphas, axis = 0)
        alphas_cumprod_prev = np.append(1., alphas_cumprod[:-1])

        # The core buffers needed for q_sample, q_posterior, and p_mean_variance
        self.sqrt_alphas_cumprod_prev = np.sqrt(np.append(1., alphas_cumprod))

        timesteps, = betas.shape
        self.num_timesteps = int(timesteps)
        self.register_buffer('betas', to_torch(betas))
        self.register_buffer('alphas_cumprod', to_torch(alphas_cumprod))
        self.register_buffer('alphas_cumprod_prev', to_torch(alphas_cumprod_prev))
        self.register_buffer('sqrt_alphas_cumprod', to_torch(np.sqrt(alphas_cumprod)))
        self.register_buffer('sqrt_one_minus_alphas_cumprod', to_torch(np.sqrt(1. - alphas_cumprod)))
        self.register_buffer('sqrt_recip_alphas_cumprod', to_torch(np.sqrt(1. / alphas_cumprod)))
        self.register_buffer('sqrt_recipm1_alphas_cumprod', to_torch(np.sqrt(1. / alphas_cumprod - 1)))

        posterior_variance = betas * (1. - alphas_cumprod_prev) / (1. - alphas_cumprod)
        self.register_buffer('posterior_variance', to_torch(posterior_variance))
        self.register_buffer('posterior_log_variance_clipped', to_torch(np.log(np.maximum(posterior_variance, 1e-20))))
        self.register_buffer('posterior_mean_coef1', to_torch(betas * np.sqrt(alphas_cumprod_prev) / (1. - alphas_cumprod)))
        self.register_buffer('posterior_mean_coef2', to_torch((1. - alphas_cumprod_prev) * np.sqrt(alphas) / (1. - alphas_cumprod)))

    def predict_start_from_noise(self, x_t, t, noise):
        return self.sqrt_recip_alphas_cumprod[t] * x_t - \
                self.sqrt_recipm1_alphas_cumprod[t] * noise


    def q_posterior(self, x_start, x_t, t):
        posterior_mean = self.posterior_mean_coef1[t] * x_start + \
                         self.posterior_mean_coef2[t] * x_t
        posterior_log_variance_clipped = self.posterior_log_variance_clipped[t]

        return posterior_mean, posterior_log_variance_clipped

    def p_mean_variance(self, x, t, clip_denoise: bool, condition_x = None):
        batch_size = x.shape[0]
        noise_level = torch.tensor(
            [self.sqrt_alphas_cumprod_prev[t + 1]], device = torch.float32
        ).repeat(batch_size, 1).to(x.device)

        if condition_x is not None:
            # Stage 1 pseudo-clean image J and the transmission map
            # It uses to predict the noise \epsilon
            # Use physical priors t make a much more accurate prediction of the noise
            noise_pred = self.denoise_fn(torch.cat([condition_x, x], dim=1), noise_level)
        else:
            noise_pred = self.denoise_fn(x, noise_level)

        # Estimate the clean image (x_0) 
        x_recon = self.predict_start_from_noise(x, t=t, noise=noise_pred)

        # Clipped for stability
        if clip_denoised:
            x_recon.clamp_(-1., 1.)

        # If we know the current noisy image and we have a good estimate of the clean image
        # The distribution of the previous step x_(t-1) is analytically solvable
        # It becomes a tractable Gaussian posteriors
        model_mean, posterior_log_variance = self.q_posterior(x_start=x_recon, x_t=x, t=t)
        return model_mean, posterior_log_variance

    @torch.no_grad()
    def p_sample(self, x, t, clip_denoised = True, condition_x = None):
        model_mean, model_log_variance = self.p_mean_variance(x = x, t = t, 
                                            clip_denoised = clip_denoised, condition_x = condition_x)

        noise = torch.randn_like(x) if t > 0 else torch.zeros_like(x)
        return model_mean + noise * (0.5 * model_log_variance).exp()